# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}\n")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

First, we'll enumerate all record set `@id`s. Then, for each record set, identify available fields and columns by `@id`.

In [ ]:
# List all record set @id's defined in the dataset
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # If there's only one, make sure it's a list
    if isinstance(metadata.recordSet, list):
        for rs in metadata.recordSet:
            record_sets.append(rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs)
    else:  # single recordSet as dict
        rs = metadata.recordSet
        record_sets.append(rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs)
else:
    print('No recordSet entities found in the metadata.')

if not record_sets:
    # Some FAIR2 datasets put record sets in distribution/fileObject. Let's try there.
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            if hasattr(dist, 'recordSet'):
                rs = dist.recordSet
                if isinstance(rs, list):
                    for r in rs:
                        if isinstance(r, dict) and '@id' in r:
                            record_sets.append(r['@id'])
                        elif isinstance(r, str):
                            record_sets.append(r)
                elif isinstance(rs, dict) and '@id' in rs:
                    record_sets.append(rs['@id'])
if not record_sets:
    # Try to see what record sets mlcroissant sees
    record_sets = [rs['@id'] for rs in dataset._metadata._find_entities('cr:RecordSet')]

print(f"Record sets (@id):\n{record_sets}\n")

for rs_id in record_sets:
    print(f"- Record set '@id': {rs_id}")
    # Find fields
    rs = dataset._metadata._find_entity_by_id(rs_id)
    field_ids = []
    column_ids = []
    if rs is not None and 'cr:field' in rs:
        fields = rs['cr:field']
        if isinstance(fields, list):
            field_ids = [fld['@id'] if isinstance(fld, dict) and '@id' in fld else fld for fld in fields]
        else:
            field_ids = [fields['@id'] if isinstance(fields, dict) and '@id' in fields else fields]
    if rs is not None and 'cr:column' in rs:
        columns = rs['cr:column']
        if isinstance(columns, list):
            column_ids = [col['@id'] if isinstance(col, dict) and '@id' in col else col for col in columns]
        else:
            column_ids = [columns['@id'] if isinstance(columns, dict) and '@id' in columns else columns]
    print(f"  Fields (@id): {field_ids}")
    print(f"  Columns (@id): {column_ids}\n")

#### Preview First Records from Each Record Set
We use the mlcroissant API directly to preview a small sample of each record set by its `@id`.

In [ ]:
for rs_id in record_sets:
    print(f"Preview of first 2 records in record set: {rs_id}")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(rec)
        if i >= 1:
            break
    print('---')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entities are referenced by their `@id`s.

In [ ]:
# Fill in your discovered record set @id(s) from above:
record_sets = record_sets  # Already obtained in overview section
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded DataFrame for record set '{record_set_id}'")
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for record set: {record_set_id}")


## 4. Exploratory Data Analysis (EDA)
We'll apply common data processing steps on the main record set. 
Identify a numeric field by its `@id`, then filter, normalize, and group the data.

_Note: Replace variables below (`numeric_field_id`, `group_field_id`) according to the overview above if needed._

In [ ]:
import numpy as np
from typing import Optional

# Choose a record set to analyze
main_record_set_id: Optional[str] = None
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f'Using record set: {main_record_set_id}')
else:
    raise Exception('No dataframes found.')

df = dataframes[main_record_set_id]

# Try to find a numeric field:
def guess_numeric_column_id(df):
    for c in df.columns:
        # heuristic: try to find one by type or well-known names
        if df[c].dtype.kind in 'iufc':
            return c
        # If type is object, try converting first 10 rows to float
        try:
            pd.to_numeric(df[c].head(10))
            return c
        except:
            continue
    return None

numeric_field_id = guess_numeric_column_id(df)
if numeric_field_id is None:
    print(f'No obvious numeric field found in DataFrame columns: {df.columns}')
    numeric_field_id = df.columns[0]  # fallback
print(f'Using numeric field @id: {numeric_field_id}')

# Try to find a plausible group field (e.g. sample, description, category)
def guess_group_field_id(df, exclude):
    for c in df.columns:
        if c != exclude and df[c].nunique() < max(10, len(df) // 10):
            return c
    return None

group_field_id = guess_group_field_id(df, numeric_field_id)
print(f'Using group field @id: {group_field_id}')

threshold = np.nanmean(pd.to_numeric(df[numeric_field_id], errors='coerce'))
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")

# Normalization (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') -
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize numeric field distribution and, if available, grouped means.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of the numeric field
vals = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
plt.figure(figsize=(7,4))
plt.hist(vals, bins=30, color='dodgerblue', alpha=0.7)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping is available, show grouped means
if group_field_id and group_field_id in df.columns:
    group_vals = (
        df[[group_field_id, numeric_field_id]]
        .groupby(group_field_id)
        .mean(numeric_only=True)
        .sort_values(numeric_field_id)
    )
    group_vals.plot(kind='bar', legend=False, figsize=(8,4), color='slateblue')
    plt.title(f"Grouped mean of {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and inspected the Mass Spectrometry protein dataset using its Croissant schema
- Explored available record sets and their fields/columns via their `@id`s
- Loaded records into DataFrames, filtered and normalized a numeric field, and grouped by a categorical field
- Visualized distributions and group statistics for further biological/proteomics insight

For more advanced usage, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/mlcroissant.python.html#mlcroissant.Dataset) and reference the dataset's Croissant schema for precise details on fields, their full `@id`, and intended semantics.